# 🧭 Cuaderno 04 — EDO: pasos múltiples y casos

**Métodos Numéricos · Unidad 5: Derivadas, integrales y EDO**

Tema **6.3 Métodos de pasos múltiples** (Adams-Bashforth, Adams-Moulton,
predictor-corrector) — capítulo 26 del libro — más un **caso de ingeniería**
que compara métodos de un paso y de pasos múltiples.

Aprendes a usar `solve_ivp(method='LSODA'|'BDF')` y `scipy.integrate.odeint`
(equivalente MATLAB `ode15s`/`ode113`), y a reconocer cuándo un problema es
**rígido**.

> Camino: **teoría → laboratorio → entradas/salidas → preguntas → caso →
> envío**.

In [ ]:
#@title ⚙️ Entorno — ejecuta esta celda PRIMERO
import os, sys

CUADERNO = 'u5_04_edo_multipaso'
TAREA_ID = 'U5_04_EDO_Multipaso'
REPO_NOMBRE = 'U5 derivadas integrales y EDO'

def _raiz_local():
    b = os.getcwd()
    for _ in range(6):
        if os.path.exists(os.path.join(b, 'lib', 'robomat.py')):
            return b
        padre = os.path.dirname(b)
        if padre == b:
            break
        b = padre
    return None

try:
    import google.colab  # noqa: F401
    ES_COLAB = True
except ImportError:
    ES_COLAB = False

if ES_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO = os.path.join('/content/drive/MyDrive', 'CursoMN', REPO_NOMBRE)
else:
    REPO = _raiz_local()
    if REPO is None:
        raise RuntimeError('Abre el cuaderno dentro de ' + REPO_NOMBRE)

sys.path.insert(0, os.path.join(REPO, 'lib'))
print('Modo       :', 'Colab' if ES_COLAB else 'local')
print('Repositorio:', REPO)

In [ ]:
#@title 📦 Librerías
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

exec(open(os.path.join(REPO, 'lib', 'robomat.py'), encoding='utf-8').read(), globals())

## 1. Teoría: métodos de pasos múltiples

A diferencia de los métodos de **un paso** (que usan solo $y_i$), los de
**pasos múltiples** usan **varios puntos previos** $y_i, y_{i-1}, y_{i-2},
\dots$ para calcular $y_{i+1}$. Por eso **no se autoinician**: necesitan
valores de arranque (los da un método de un paso, típicamente RK4).

### Adams-Bashforth (explícito / abierto)

$$y_{i+1} = y_i + \frac{h}{24}\left(55f_i - 59f_{i-1} + 37f_{i-2} - 9f_{i-3}\right) \tag{26.40}$$

### Adams-Moulton (implícito / cerrado)

$$y_{i+1} = y_i + \frac{h}{24}\left(9f_{i+1} + 19f_i - 5f_{i-1} + f_{i-2}\right) \tag{26.45}$$

El **predictor-corrector** combina ambos: Bashforth predice, Moulton corrige.

### Sistemas rígidos

Una EDO es **rígida** si tiene componentes que cambian a escalas muy distintas
(una rapidísima y otra lenta). Los métodos explícitos exigen un paso $h$
diminuto para no explotar; los **implícitos** (BDF) lo manejan sin colapsar.

> 📖 **Figuras:** `fig_26_6.pdf` (pasos previos), `fig_26_7.pdf`
> (Newton-Cotes vs Adams).

## 2. Laboratorio: multipaso y rigidez con `solve_ivp`

Resuelve el problema **rígido** clásico (cap. 26):

$$y' = -1000y + 3000 - 2000e^{-t}, \qquad y(0)=0$$

cuya solución exacta es $y = 3 - 0.998e^{-1000t} - 2.002e^{-t}$. El término
$e^{-1000t}$ decae casi al instante, pero obliga a pasos muy pequeños a los
métodos explícitos.

In [ ]:
def rigido(t, y):
    return -1000*y + 3000 - 2000*np.exp(-t)

def exacta(t):
    return 3 - 0.998*np.exp(-1000*t) - 2.002*np.exp(-t)

# BDF (método implícito multipaso, para rígidas)
sol_bdf = solve_ivp(rigido, (0, 0.1), [0.0], method='BDF')
# RK45 (explícito): con h por omisión puede tardar/fallar en rígidas
try:
    sol_rk = solve_ivp(rigido, (0, 0.1), [0.0], method='RK45', max_step=1e-4)
    rk_ok = True
except Exception:
    rk_ok = False

print(f'BDF  : y(0.1) = {sol_bdf.y[0][-1]:.6f}  (exacta {exacta(0.1):.6f})')
if rk_ok:
    print(f'RK45 : y(0.1) = {sol_rk.y[0][-1]:.6f}  (con max_step=1e-4)')
else:
    print('RK45 no terminó con paso por omisión: el problema es rígido.')

## 3. Entradas, salidas y cuándo usar cada método

### `solve_ivp(..., method='LSODA')`

- Cambia **automáticamente** entre Adams (no rígida) y BDF (rígida) según el
  problema. Buena elección por omisión cuando no sabes si es rígido.

### `solve_ivp(..., method='BDF')`

- Método **implícito de pasos múltiples** (Backward Differentiation Formulas),
  pensado para **sistemas rígidos**. Más robusto, algo más costoso por paso.

### `scipy.integrate.odeint(func, y0, t)`

- API clásica (envuelve LSODA): recibe `func(y, t)` (orden invertido),
  `y0` y el arreglo `t` de salida. Devuelve el arreglo de soluciones.

> 🐍 **MATLAB:** `ode113` (Adams-Bashforth-Moulton), `ode15s` (rígidas, BDF),
> `ode23s` (rígidas).

**Regla práctica:** si la solución cambia suavemente usa RK45; si ves que el
paso se desploma o hay una escala rápida escondida, cambia a `BDF`/`LSODA`.

## 4. Preguntas teóricas

**P1.** ¿Por qué un método de pasos múltiples **no se autoinicia**?

1. Porque requiere valores de varios puntos previos que aún no existen.
2. Porque solo funciona con paso constante.
3. Porque es implícito y necesita resolver un sistema.
4. Porque no puede usar condiciones iniciales.

**P2.** Un sistema rígido se caracteriza por…

1. ser siempre lineal y de primer orden.
2. tener componentes con escalas de tiempo muy distintas.
3. tener más de una variable independiente.
4. no tener solución analítica.

In [ ]:
RESPUESTA_P1 = 1   # necesita puntos previos
RESPUESTA_P2 = 2   # escalas de tiempo muy distintas

## 5. Caso de ingeniería: comparar métodos

Un circuito RC con una constante de tiempo muy corta genera una EDO rígida.
Compara el **costo** (número de evaluaciones) y la **precisión** de `RK45`
(un paso) contra `BDF` (multipaso) al resolver el problema rígido de arriba,
y concluye cuál conviene.

In [ ]:
def rigido(t, y):
    return -1000*y + 3000 - 2000*np.exp(-t)

sol_bdf = solve_ivp(rigido, (0, 0.1), [0.0], method='BDF')
sol_rk  = solve_ivp(rigido, (0, 0.1), [0.0], method='RK45', max_step=1e-4)

def exacta(t):
    return 3 - 0.998*np.exp(-1000*t) - 2.002*np.exp(-t)

print('Método | evaluaciones de f | y(0.1)       | error')
print(f'BDF    | {sol_bdf.nfev:<17} | {sol_bdf.y[0][-1]:.6f} | {abs(sol_bdf.y[0][-1]-exacta(0.1)):.2e}')
print(f'RK45   | {sol_rk.nfev:<17} | {sol_rk.y[0][-1]:.6f} | {abs(sol_rk.y[0][-1]-exacta(0.1)):.2e}')

# Conclusión: BDF logra precisión con MUCHAS menos evaluaciones en un sistema rígido.
RESPUESTA_CASO = 'BDF'

## 6. Revisar, enviar y ver calificación

Las respuestas (`RESPUESTA_P1`, `RESPUESTA_P2`, `RESPUESTA_CASO`) se envían
al **Apps Script** (se conecta en la Fase 4).

In [ ]:
#@title 📤 Enviar respuestas y ver calificación
alumno_id = 'TU_NUMERO_CONTROL'
respuestas = {'P1': RESPUESTA_P1, 'P2': RESPUESTA_P2, 'CASO': RESPUESTA_CASO}
print('Pendiente de conectar con Apps Script:', respuestas)